<a href="https://colab.research.google.com/github/mikanogami/MIE1517-Team25/blob/main/Final_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Overview

Let’s say you want to navigate a robot through a maze.  This is typically done by hard coding robot actions based on sensor inputs. This method requires extensive tuning and testing with the robot to ensure that the hard coded actions cover all robot states including edge cases.

Our project aims to train a neural network navigates a mobile robot around a maze environment in simulation. The mobile robot moves at constant speed, and at each timestep of the simulator, the model takes ultrasonic sensor data (representing the robot state) as an input and outputs the steering angle of the mobile robot. By training a neural network to output a robot action based on ultrasonic sensor data, we remove the need for hard coded robot actions. Our agent is a fully-connected neural network with one hidden layer that is trained using DAgger, a common imitation learning algorithm. Imitation learning is a subset of reinforcement learning where the agent (our model) learns to perform a task from expert demonstrations.

We implemented and tested the model on the simmer-python simulator that was used for MIE444 at UofT.

The GIF below shows an agent navigating a maze that it was not trained on.

In [ ]:
from IPython.display import Image
Image(open('notebook/mazedemo.gif','rb').read())

#Note: video is sped up for the sake of reducing filesize

# Simulator

We are using a mobile robot simulator called [simmer-python](https://github.com/UToronto-MIE444/simmer-python) that was developed as a teaching tool for MIE444 Mechatronics Principles at the University of Toronto. The simulator is open-source and was developed for educational purposes.

Note: WE RECOMMEND RUNNING THIS DEMO LOCALLY.  
The pygame simulation will not open in Google Colab.

In [ ]:
# We recommend running this demo locally because the pygame simulation will not show up using colab
use_colab=False 

if use_colab:
    %cd /content
    !rm -rf MIE1517-Team25
    !git clone https://github.com/mikanogami/MIE1517-Team25
    %cd MIE1517-Team25

import numpy as np
import pygame
import sys, os
import csv
import datetime, time
import torch
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


sys.path.append(os.path.abspath("simmer-python"))
from maze import Maze
from robot import Robot
from block import Block
from interface.hud import Hud
from interface.communication import TCPServer
import config as CONFIG
import utilities
import main
import simmer

from expert import Expert
from net import RobotControlNet

You can run the mobile robot simulation (using below command) and teleoperate the robot using the following keypress commands:
W/S: move forward/backward  
Q/E: move right/left  
D/A: rotate right/left  

In [ ]:
!python simmer-python/simmer.py

simmer-python/config.py file determines the properties of the mobile robot, its sensors, and the maze. We can change the configuration of the robot and the maze by updating the **CONFIG** object from within config.py.

CONFIG.walls: defines the configuration for maze walls  
CONFIG.sensors: defines all sensor types, placements, and errors  

###**Set up Sensors** *simmer-python/config.py*

Define the sensors on the mobile robot in *config.py*. 

Our model used 5 ultrasonic sensors which we defined as u0, u1, u2, u3, u4 in *config.py*. 
- u1, u2, u3 are front-facing and are mounted at the left, middle, and right at the front edge of the robot
- u0 is mounted facing left and is mounted at the left front corner of the robot
- u4 is mounted facing right and is mounted at the right front corner of the robot

Note: THIS IS THE SENSOR CONFIGURATION WE WILL BE USING IN THIS DEMO.

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(10, 5))

# First image
img1 = mpimg.imread('notebook/robot_sensors.png')
axes.imshow(img1)
axes.axis('off')  # Hide axes

plt.tight_layout()
plt.show()

An example of how ultrasonic sensors are defined in *simmer-python/config.py* is shown below.

In [ ]:
import pygame.math as pm
from devices.ultrasonic import Ultrasonic

# define sensor configuration
u0_info = {
    'id': 'u0',
    'position': [3, 3],
    'height': 2,
    'rotation': 0,
    'error': 0.02,
    'outline': [
        pm.Vector2(-1, -0.5),
        pm.Vector2(-1, 0.5),
        pm.Vector2(1, 0.5),
        pm.Vector2(1, -0.5)
    ],
    'visible': True,
    'visible_measurement': True
}

# add u0 to list of sensors in config
sensors = {
    'u0': Ultrasonic(u0_info),
    # other sensors are listed here
}

# add u0 to simulate_list if you want it to show up in the simulation
simulate_list = ['u0']

# Data Collection

**DAgger (Dataset Aggregation)**

DAgger, short for Dataset Aggregation, is a common imitation learning algorithm. The model is trained iteratively on data collected from the expert, which controls the robot initially around the track. With this data, the first model is created and then deployed such that it becomes the agent to drive the robot around the track. When the robot veers too far off the desired trajectory, the expert corrects the robot and the data from the expert correcting the robot gets appended to the existing training data. This aggregated training data with more information is then used to train another iteration of the model. This process repeats until the robot is sufficient to drive without the expert taking over.

**Setup Training Environment** *simmer-python/config.py*

First, set up a simple track that we will deploy the robot on to collect training data. The training environment we chose was a rectangular track. We chose a simple training environment because we needed to define a trajectory for the **EXPERT** to collect robot data in that training environment. The pre-defined reference trajectory for this environment is a straight path in the middle of the maze corridor for the straight sections of the track and a circular trajectory with a turning radius of one-half the width of the maze corridor around the corners. The reference trajectory for our training environment is show in the images below in red for both the clockwise and counterclockwise cases.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# First image
img1 = mpimg.imread('notebook/expert_trajectory_CCW.jpeg')
axes[0].imshow(img1)
axes[0].set_title("Expert Reference Trajectory Counterclockwise")
axes[0].axis('off')  # Hide axes

# Second image
img2 = mpimg.imread('notebook/expert_trajectory_CW.jpeg')
axes[1].imshow(img2)
axes[1].set_title("Expert Reference Trajectory Clockwise")
axes[1].axis('off')

plt.tight_layout()
plt.show()

The maze object defined in *simmer-python/config.py* that corresponds to our training environment is:

walls = [[0,0,0,0,0,0,0,0],
         [0,1,1,1,1,1,1,0],
         [0,1,1,1,1,1,1,0],
         [0,0,0,0,0,0,0,0]]

**Creating the Expert** *expert.py*

In imitation learning, the agent (our model) learns to perform a task from expert demonstrations. For the simulated mobile robot, an example of an expert is a human operator, steering the robot using keypresses. We chose to implement the **EXPERT** as a controller that uses PID control to keep the robot moving along a set trajectory. The reference trajectory that the expert follows in our training environment is shown in the images above.


**Dataset**
The expert controls the robot as it navigates the training environment. At each timestep, we record the ultrasonic sensor readings (model inputs) and the corresponding steering command from the expert (labels used for supervised learning). 

Our model is trained on our initial dataset and our aggregated dataset:
1) Initial dataset: expert steers the robot around the track in clockwise and counterclockwise directions. This dataset is a distribution of robot states that the expert is likely to experience. Therefore, most of the robot states recorded are when the robot is not deviating very much from the reference trajectory.

2) Aggregated dataset: the agent is trained on the initial dataset and deployed to steer the robot around the track. At each timestep, we record the heading error (difference in angle between reference trajectory heading and robot heading) and cross-track error (the perpendicular distance between the robot and the reference trajectory). If the heading error or cross-track error pass a certain threshold while the agent is controlling the robot, the expert will take over from the agent to correct the robot trajectory. Once the expert has corrected the robot course, the agent takes over again. Data is collected while the expert is controlling the robot to correct the agent's steering commands and is aggregated to the existing dataset. The agent is retrained on the aggregated dataset and deployed on the maze again (DAgger algorithm).

In [ ]:
from main import run_sim
import multiprocessing

def run_CW_track(model, dagger_itr, save_dir, runtime=120):
    # run simulation with robot moving clockwise
    start_pos_CW = [6, 36]
    start_rot_CW = 180
    process = multiprocessing.Process(target=run_sim, args=(model, dagger_itr, runtime, save_dir, start_pos_CW, start_rot_CW, True))
    process.start()
    process.join()

def run_CCW_track(model, dagger_itr, save_dir, runtime=120):
    # run simulation with robot moving counter clockwise
    start_pos_CCW = [6, 12]
    start_rot_CCW = 0
    process = multiprocessing.Process(target=run_sim, args=(model, dagger_itr, runtime, save_dir, start_pos_CCW, start_rot_CCW, False))
    process.start()
    process.join()


We get our INITIAL DATASET by running the robot simulator with model=None. When model is set to None, the expert will control the robot the entire time the simulation is running and all that data will be saved as our initial dataset.

In [ ]:
# Note: set save_dir to the directory that you want the data saved to; save_dir=None means data will not be saved
run_CW_track(model=None, dagger_itr=0, save_dir=None, runtime=60)
run_CCW_track(model=None, dagger_itr=0, save_dir=None, runtime=60)

Below is the distribution of training data over the classes of our classification model which correspond to steering angle commands for the robot. As you can see, a large part of the data corresponds to the straight steering angle class. We show data distribution over steering angle classes for our initial dataset, after 1 iteration of DAgger, and after 2 iterations of DAgger.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 5))

# First image
img1 = mpimg.imread('notebook/datadistribution_0dagger.png')
axes[0].imshow(img1)
#axes[0].set_title("Expert Reference Trajectory Counterclockwise")
axes[0].axis('off')  # Hide axes

# Second image
img2 = mpimg.imread('notebook/datadistribution_1dagger.png')
axes[1].imshow(img2)
#axes[1].set_title("Expert Reference Trajectory Clockwise")
axes[1].axis('off')

# Second image
img2 = mpimg.imread('notebook/datadistribution_2dagger.png')
axes[2].imshow(img2)
#axes[1].set_title("Expert Reference Trajectory Clockwise")
axes[2].axis('off')

plt.tight_layout()
plt.show()

Due to the prominance of data with a "straight" steering angle corresponding to class 7 in the above plots, the difference in data distributions is not noticeable. Below are plots of the same data distributions over DAgger iterations, omitting class 7. Here we can see that during the first DAgger iteration, data is collected corresponding to steering classes 0-6 and 8-15, making the overall data distribution more uniform. We do not see a significant difference between data distributions after the first and second DAgger iterations. This is because after the first iteration of DAgger, our model is able to navigate the training environment with very little expert intervention, meaning that very little data is added to our dataset.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 5))

# First image
img1 = mpimg.imread('notebook/datadist_0dagger_nostraight.png')
axes[0].imshow(img1)
#axes[0].set_title("Expert Reference Trajectory Counterclockwise")
axes[0].axis('off')  # Hide axes

# Second image
img2 = mpimg.imread('notebook/datadist_1dagger_nostraight.png')
axes[1].imshow(img2)
#axes[1].set_title("Expert Reference Trajectory Clockwise")
axes[1].axis('off')

# Second image
img2 = mpimg.imread('notebook/datadist_2dagger_nostraight.png')
axes[2].imshow(img2)
#axes[1].set_title("Expert Reference Trajectory Clockwise")
axes[2].axis('off')

plt.tight_layout()
plt.show()

###**Dataloader and UniformBatchSampler** *data_loader.py*

Our data distribution over classes (binned steering angles) that the distribution is highly unbalanced. Data corresponding to class 7, the "straight" steering angle case dominates the entire dataset. It is important for smooth training that each batch of data used for training consists of data from all classes, not just from the straight steering angle class.

As such, we were motivated to create a Pytorch custom sampler that ensured that each batch loaded from the data loader had approximately the same distribution over classes.

In [ ]:
from torch.utils.data import DataLoader
from data_loader import SensorDataset, UniformBatchSampler, collate_fn

n_classes = 16
n_sensors = 5
batch_size = 256

train_dataset = SensorDataset(root_dir="data/projectmodel", num_classes=n_classes)
train_sampler = UniformBatchSampler(train_dataset.data, batch_size=batch_size)
train_loader = DataLoader(train_dataset, sampler=train_sampler, batch_size=None, collate_fn=collate_fn)

# test that UniformBatchSampler samples batches with approximately the same distribution of classes across batches
for i_batch, batch in enumerate(train_loader):
    data, steering_angles = batch
    cmd_counts = torch.bincount(steering_angles, minlength=n_classes)
    print('Batch {} has distribution: {}'.format(i_batch, cmd_counts))

###**Set Up Main File** *main.py*

Instantiate the expert and its respective PID controllers for track and trajectory.

In [ ]:
Insert code

Within the *while running* loop, similate the sensors.

Obtain errors from the expert and calculate the track and trajectory steering angle adjustment. The total adjustment is the sum of both. Smooth the total adjustment with. variable factor named alpha, to blend the previous and current update.


Collect sensor data and save it to a csv.

###**Main Loop** *main.py*

Set up the robot's starting position and orientation for the clockwise direction. Simulate the robot going in this direction.

Set up the robot's starting position and orientation for the clockwise direction. Simulate the robot going in this direction


###**Tune PID and Alpha** *main.py*

Play around with the K values for the PID controllers and the alpha value for blending the steering adjustment updates such that the expert can drive the robot around the track smoothly and it will not get stuck in the walls. These values may need to be modified as you perform the DAgger and the robot experiences different pertubations that the expert needs to correct.

###**Collect Data** *main.py*

Simply run the main file to collect sensor data. There will be 2 csv's in the data folder, one for the clockwise direction and one for the counterclockwise direction.



# Model Architecture

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

img = mpimg.imread('Model Architecture.png')  # path to your image
plt.imshow(img)
plt.axis('off')  # hide axis
plt.show()

**Model Definition**

Our chosen model is quite simple, with just 2 fully connected layers. The input size was tested and varied with the number of ultrasonic sensors, while the output size could also change depending on the number of classes chosen. These classes correspond to the discrete bins for steering angle adjustment.


In [ ]:
class RobotControlNet(nn.Module):
  pass

# DAgger Implementation

DAgger, short for Dataset Aggregation, is a common imitation learning algorithm. The model is trained iteratively on data collected from the expert, which controls the robot initially around the track. With this data, the first model is created and then deployed such that it becomes the agent to drive the robot around the track. When the robot veers too far off the desired trajectory, the expert corrects the robot and the data from the expert correcting the robot gets appended to the existing training data. This aggregated training data with more information is then used to train another iteration of the model. This process repeats until the robot is sufficient to drive without the expert taking over.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

img = mpimg.imread('End_to_end_pipeline.png')  # path to your image
plt.imshow(img)
plt.axis('off')  # hide axis
plt.show()

###**Create Dataset** *dataloader.py*

Make a custom class for the dataset to read the csv files and extract the desired features.

In [ ]:
class SensorDataset(Dataset):
  pass

###**Pre-Processing** *net.py*
Create a uniform batch sampler that goes over all the data to make sure each batch loaded during training has approximately the same distribution over the classes.

In [ ]:
class UniformBatchSampler(Sampler):
  pass

###**Dataset Visualization** */data/*
Below we can see what the training data looks like after it was collected by the expert. The first 5 columns are the ultrasonic sensor data and the last column is the steering adjustment that the expert outputs. The sensor data constitutes the inputs to our model, while the steering adjustment denotes the labels.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

column_names = ['u0', 'u1', 'u2', 'u3', 'u4', 'steering_adj']
data = pd.read_csv('/content/training_data_0_CW.csv', header=None, names=column_names)
print(data)

From the histogram below, it is clear to see that the robot has a lot more data corrosponding to the smaller steering adjustments, likely due to the greater duration of time spent travelling straight.

In [ ]:
# Insert histogram of data distribution

###**Train Policy** *train_policy.py*
To train the the model, the code is very similar to other models we have trained in class. The only deviation from the normal code is when we weight the errors according to the inverse frequency of occurance. This prevents the model from overfitting to data that is unequally distributed across classes.

In [ ]:
def train_model(model, train_loader, dagger_itr, learning_rate=1e-3, num_epochs=100):

###**Deploy Model** *main.py*
Add logic to toggle between using the model to adjust the steering angle and the expert depending on if the either the cross track error or the heading error is greater than threshold deviation than desired. If the expert corrects the robot's movement, append those lines of the sensor data to the training dataset.

###**Observe Result**
After deploying the model, watch the robot go around the track.

In [ ]:
# Video of model driving track-

###**Repeat Process**
Keep re-training the model with more iterations until you are happy with the result.

# Results on Training Environment

###**Quantitative Results**

For quantitative results, we measured the training error and Cumulative Cross Track Error throughout training. We found that CTE decreased with DAgger iterations. Training error consistently stayed at ~0.35, but this can partially be atrributed to the fact that our model makes categorical predictions. In reality, the difference between a 17 $^{\circ}$ and 18 $^{\circ}$ turning angle is not significant, but this would be logged as an incorrect prediction.

###**Qualitative Results**

Qualitatively it can be observed that with each DAgger iteration, less expert intervention is required. By the third DAgger iteration, no intervention is needed at all.

# Results on Unseen Environment

```
Code to implement on new data
```

###**Quantitative Results**

###**Qualitative Results**

## Summary of Related Work

include citations/references
Results on caomparable projects, links to publications etc

# Discussion

**Is our model performing well? why or why not?**
- yes, the model can navigate a maze it was not trained on
- compared to expert that needs direct trajectory definition - can generalize to other maze shapes

**What is unusual, suprisign or interesting about your results? what did you learn**

**Any great insights from working on this project?**
- PID tuning was pain

## Limitations of our Method

Our method does great on unseen environments consisting of right and left turns because our simple training environment track consisted of only right and left turns. However, the robot struggles in environments containing branching or dead ends.